## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-18-13-50-50 +0000,nyt,"Live Updates: Trump, Xi and Putin Shake Hands ...",https://www.nytimes.com/live/2026/11/18/world/...
1,2026-11-18-13-50-08 +0000,wapo,"Hegseth survives impeachment fight, but more p...",https://www.washingtonpost.com/national-securi...
2,2026-11-18-13-47-46 +0000,nyt,Irish Man Sabotaged U.S. Military Plane With A...,https://www.nytimes.com/2026/11/18/world/europ...
3,2026-11-18-13-46-45 +0000,nyt,Leaders of China and Japan Meet Amid Political...,https://www.nytimes.com/2026/11/18/world/asia/...
4,2026-11-18-13-45-00 +0000,wsj,Oil Prices Fall Amid Supply Surplus Expectatio...,https://www.wsj.com/business/energy-oil/oil-pr...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2748/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
2,trump,72
108,nvidia,32
293,new,23
9,china,22
202,huang,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
257,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...,178
386,2026-11-17-18-34-00 +0000,wsj,A lawyer for Fed governor Lisa Cook said she w...,https://www.wsj.com/politics/policy/feds-lisa-...,169
39,2026-11-18-12-21-51 +0000,cbc,"Xi, Putin meet in China, ahead of summit with ...",https://www.cbc.ca/news/world/shenzen-jinping-...,149
33,2026-11-18-12-39-24 +0000,bbc,Trump shakes hands with Putin and Xi ahead of ...,https://www.bbc.co.uk/news/articles/ck9d75x5ww...,148
44,2026-11-18-12-07-00 +0000,wsj,"In Choosing Shenzhen for Trump-Putin Summit, X...",https://www.wsj.com/world/china/in-choosing-sh...,140


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
257,178,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...
386,97,2026-11-17-18-34-00 +0000,wsj,A lawyer for Fed governor Lisa Cook said she w...,https://www.wsj.com/politics/policy/feds-lisa-...
27,84,2026-11-18-12-56-00 +0000,wsj,The Morning Risk Report: SEC Is Probing Nvidia...,https://www.wsj.com/risk-compliance-journal/th...
353,84,2026-11-17-20-01-00 +0000,wsj,Progressive activists say the Senate next year...,https://www.wsj.com/politics/policy/the-angry-...
179,61,2026-11-18-04-23-00 +0000,wsj,Democrat Mary Peltola unseated Republican Sen....,https://www.wsj.com/politics/elections/democra...
346,59,2026-11-17-20-31-11 +0000,nyt,Ukraine Talks Run Up Against Deep Enmity Betwe...,https://www.nytimes.com/2026/11/17/us/politics...
426,58,2026-11-17-17-00-56 +0000,chitri,Thanksgiving travel plans? AAA predicts fewer ...,https://www.chicagotribune.com/2026/11/17/than...
54,58,2026-11-18-11-57-00 +0000,wsj,Homeland Security Secretary Markwayne Mullin h...,https://www.wsj.com/politics/policy/markwayne-...
276,57,2026-11-17-22-33-00 +0000,wsj,"Hung Cao, President Trump’s pick to lead the N...",https://www.wsj.com/politics/national-security...
44,51,2026-11-18-12-07-00 +0000,wsj,"In Choosing Shenzhen for Trump-Putin Summit, X...",https://www.wsj.com/world/china/in-choosing-sh...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
